# GeoDjango Integration for Census Boundaries

This notebook demonstrates the GeoDjango integration module for storing and querying
Census geographic boundaries in a PostGIS database.

## Prerequisites

1. PostgreSQL with PostGIS extension
2. Django project configured with GeoDjango
3. siege_utilities installed with geodjango extras:
   ```bash
   pip install siege-utilities[geodjango]
   ```

## Contents

1. Module Overview
2. Model Structure
3. Django Configuration
4. Management Commands
5. Spatial Queries
6. Working with Demographics
7. Crosswalk Queries

## 1. Module Overview

The GeoDjango module provides:

- **Models**: State, County, Tract, BlockGroup, Block, Place, ZCTA, CongressionalDistrict
- **Services**: Boundary, Demographic, and Crosswalk population services
- **Serializers**: DRF GeoJSON serializers for API responses
- **Management Commands**: CLI tools for populating data

In [ ]:
import sys
from pathlib import Path

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

# Check module availability
from siege_utilities.geo.django import _django_available, _gis_available, __version__

su.log_info(f"GeoDjango Module Version: {__version__}")
su.log_info(f"Django Available: {_django_available}")
su.log_info(f"GeoDjango (GIS) Available: {_gis_available}")

In [ ]:
# --- Path configuration ---
from siege_utilities.config.user_config import get_user_config

DATA_DIR = Path(get_user_config().get_download_directory())
DATA_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Data directory: {DATA_DIR}  (exists={DATA_DIR.exists()})")

## 2. Model Structure

### GEOID Formats

| Geography | GEOID Length | Format | Example |
|-----------|--------------|--------|----------|
| State | 2 | SS | 06 (California) |
| County | 5 | SSCCC | 06037 (Los Angeles) |
| Tract | 11 | SSCCCTTTTTT | 06037101100 |
| Block Group | 12 | SSCCCTTTTTTB | 060371011001 |
| Block | 15 | SSCCCTTTTTTBBBB | 060371011001001 |
| Place | 7 | SSPPPPP | 0644000 (Los Angeles city) |
| ZCTA | 5 | ZZZZZ | 90210 |
| CD | 4 | SSDD | 0614 (CA-14) |

In [ ]:
# Example: Parse GEOIDs (this works without Django)
def parse_tract_geoid(geoid: str) -> dict:
    """Parse a tract GEOID into components."""
    return {
        "state_fips": geoid[:2],
        "county_fips": geoid[2:5],
        "tract_code": geoid[5:11],
    }

# Test with a California tract
geoid = "06037101100"
parsed = parse_tract_geoid(geoid)
su.log_info(f"GEOID: {geoid}")
su.log_info(f"State FIPS: {parsed['state_fips']} (California)")
su.log_info(f"County FIPS: {parsed['county_fips']} (Los Angeles)")
su.log_info(f"Tract Code: {parsed['tract_code']}")
su.log_info(f"Tract Number: {int(parsed['tract_code'][:4])}.{parsed['tract_code'][4:]}")

## 3. Django Configuration

Add the following to your Django settings:

```python
# settings.py

INSTALLED_APPS = [
    # ... other apps
    'django.contrib.gis',
    'rest_framework',
    'rest_framework_gis',
    'siege_utilities.geo.django',  # Add this
]

DATABASES = {
    'default': {
        'ENGINE': 'django.contrib.gis.db.backends.postgis',
        'NAME': 'your_database',
        'USER': 'your_user',
        'PASSWORD': 'your_password',
        'HOST': 'localhost',
        'PORT': '5432',
    }
}
```

Then run migrations:

```bash
python manage.py makemigrations siege_geo
python manage.py migrate
```

## 4. Management Commands

### Populate Boundaries

```bash
# Populate all states
python manage.py populate_boundaries --year 2020 --type state

# Populate counties for California
python manage.py populate_boundaries --year 2020 --type county --state CA

# Populate tracts for California (requires state)
python manage.py populate_boundaries --year 2020 --type tract --state 06

# Populate with parent linking
python manage.py populate_boundaries --year 2020 --type tract --state CA --link-parents
```

### Populate Demographics

```bash
# Populate income data for California tracts
python manage.py populate_demographics --year 2022 --type tract --state CA --variables income

# Populate all variable groups
python manage.py populate_demographics --year 2022 --type tract --state CA --variables all
```

### Populate Crosswalks

```bash
# Populate 2010-2020 tract crosswalk for California
python manage.py populate_crosswalks --source-year 2010 --target-year 2020 --type tract --state CA
```

## 5. Spatial Queries

The custom `BoundaryManager` provides convenient spatial query methods.

In [ ]:
# Example spatial queries (requires Django environment)
# These are the queries you would run in a Django project:

example_queries = '''
from django.contrib.gis.geos import Point
from siege_utilities.geo.django.models import Tract, County

# Find tract containing a point (San Francisco)
sf_point = Point(-122.4194, 37.7749, srid=4326)
tract = Tract.objects.containing_point(sf_point).for_year(2020).first()
su.log_info(f"Tract: {tract.name} ({tract.geoid})")

# Find all tracts in Los Angeles County
la_tracts = Tract.objects.for_year(2020).for_state('06').filter(county_fips='037')
su.log_info(f"LA County tracts: {la_tracts.count()}")

# Find counties intersecting a polygon
from django.contrib.gis.geos import Polygon
bbox = Polygon.from_bbox((-122.5, 37.5, -122.0, 38.0))
bay_area_counties = County.objects.for_year(2020).intersecting(bbox)

# Order tracts by land area
largest_tracts = Tract.objects.for_year(2020).for_state('06').by_land_area(ascending=False)[:10]
'''
su.log_info(example_queries)

## 6. Working with Demographics

Demographics are stored in `DemographicSnapshot` models linked to boundaries via generic foreign keys.

In [ ]:
# Example demographic queries (requires Django environment)
example_demographics = '''
from siege_utilities.geo.django.models import Tract, DemographicSnapshot
from django.contrib.contenttypes.models import ContentType

# Get demographic snapshot for a tract
tract = Tract.objects.get(geoid='06037101100', census_year=2020)
ct = ContentType.objects.get_for_model(Tract)

snapshot = DemographicSnapshot.objects.get(
    content_type=ct,
    object_id=tract.geoid,
    year=2022,
    dataset='acs5'
)

# Access values
population = snapshot.get_value('B01001_001E')
income = snapshot.get_value('B19013_001E')
income_moe = snapshot.get_moe('B19013_001E')

su.log_info(f"Population: {population}")
su.log_info(f"Median Income: ${income:,} (+/- ${income_moe:,})")

# Use summary fields
su.log_info(f"Quick access: Population={snapshot.total_population}, Income=${snapshot.median_household_income}")
'''
su.log_info(example_demographics)

## 7. Crosswalk Queries

Crosswalks map boundaries between Census years (e.g., 2010 to 2020).

In [ ]:
# Example crosswalk queries (requires Django environment)
example_crosswalk = '''
from siege_utilities.geo.django.models import BoundaryCrosswalk

# Find 2020 equivalents for a 2010 tract
old_geoid = '06037101100'  # 2010 tract
mappings = BoundaryCrosswalk.get_forward_mappings(
    geoid=old_geoid,
    source_year=2010,
    target_year=2020
)

for m in mappings:
    su.log_info(f"{m.source_geoid} -> {m.target_geoid}: {m.weight:.2%} ({m.relationship})")

# Allocate a value from 2010 to 2020
old_population = 5000
for m in mappings:
    allocated = m.allocate_value(old_population)
    su.log_info(f"  -> {m.target_geoid}: {allocated:.0f} people")

# Find unchanged tracts
unchanged = BoundaryCrosswalk.objects.filter(
    geography_type='tract',
    source_year=2010,
    target_year=2020,
    relationship='IDENTICAL'
)
su.log_info(f"Unchanged tracts in CA: {unchanged.filter(state_fips='06').count()}")
'''
su.log_info(example_crosswalk)

## 8. API Serialization

Use the DRF serializers to create GeoJSON API responses.

In [ ]:
# Example API view (requires Django environment)
example_api = '''
# views.py
from rest_framework import viewsets
from siege_utilities.geo.django.models import Tract
from siege_utilities.geo.django.serializers import TractSerializer

class TractViewSet(viewsets.ReadOnlyModelViewSet):
    """API endpoint for Census Tracts."""
    queryset = Tract.objects.all()
    serializer_class = TractSerializer
    
    def get_queryset(self):
        qs = super().get_queryset()
        
        # Filter by year
        year = self.request.query_params.get('year')
        if year:
            qs = qs.for_year(int(year))
        
        # Filter by state
        state = self.request.query_params.get('state')
        if state:
            qs = qs.for_state(state)
        
        return qs

# urls.py
from rest_framework.routers import DefaultRouter

router = DefaultRouter()
router.register('tracts', TractViewSet)
urlpatterns = router.urls
'''
su.log_info(example_api)

## Next Steps

1. Set up PostGIS database
2. Configure Django project with GeoDjango
3. Run migrations to create tables
4. Use management commands to populate data
5. Build API endpoints using serializers

See also:
- [Django GeoDjango Tutorial](https://docs.djangoproject.com/en/4.2/ref/contrib/gis/tutorial/)
- [DRF GeoJSON](https://django-rest-framework-gis.readthedocs.io/)